# Assignment Beta - Question 4 (Student ID: 23I-0124)
LLM sentiment classification: zero-shot vs improved prompt. All parts 1-8 are calculated below with plain Python/NumPy.

In [1]:
import numpy as np, math
from collections import Counter
labels=['Positive','Negative','Neutral']
S1=np.array([[100,5,15],[8,80,12],[20,10,50]],float)
S2=np.array([[106,4,10],[6,84,10],[9,7,64]],float)

## Parts 1-3 and 5: metrics function
Accuracy = correct / total. 95% CI = acc ± 1.96·sqrt(acc(1-acc)/n). Precision = TP/column total, Recall = TP/row total, F1 = 2PR/(P+R). Macro = simple mean, Weighted = weighted by support, Micro = accuracy.

In [2]:
def evaluate(C,name):
    n=C.sum(); tp=np.diag(C)
    P=tp/C.sum(0); R=tp/C.sum(1); F=2*P*R/(P+R); sup=C.sum(1)
    acc=tp.sum()/n; se=math.sqrt(acc*(1-acc)/n); lo,hi=acc-1.96*se,acc+1.96*se
    print(f'--- {name} ---')
    print(f'Accuracy = {int(tp.sum())}/{int(n)} = {acc:.4f}, SE = {se:.4f}, 95% CI = [{lo:.4f}, {hi:.4f}]')
    for i,l in enumerate(labels): print(f'{l:9s} P={P[i]:.4f} R={R[i]:.4f} F1={F[i]:.4f} support={int(sup[i])}')
    macro=F.mean(); weighted=(F*sup).sum()/n; micro=acc
    print(f'Macro F1={macro:.4f}  Weighted F1={weighted:.4f}  Micro F1={micro:.4f}')
    return dict(acc=acc,lo=lo,hi=hi,P=P,R=R,F=F,macro=macro,weighted=weighted,micro=micro)
r1=evaluate(S1,'Stage 1 (zero-shot)')
r2=evaluate(S2,'Stage 2 (improved prompt)')

--- Stage 1 (zero-shot) ---
Accuracy = 230/300 = 0.7667, SE = 0.0244, 95% CI = [0.7188, 0.8145]
Positive  P=0.7812 R=0.8333 F1=0.8065 support=120
Negative  P=0.8421 R=0.8000 F1=0.8205 support=100
Neutral   P=0.6494 R=0.6250 F1=0.6369 support=80
Macro F1=0.7546  Weighted F1=0.7659  Micro F1=0.7667
--- Stage 2 (improved prompt) ---
Accuracy = 254/300 = 0.8467, SE = 0.0208, 95% CI = [0.8059, 0.8874]
Positive  P=0.8760 R=0.8833 F1=0.8797 support=120
Negative  P=0.8842 R=0.8400 F1=0.8615 support=100
Neutral   P=0.7619 R=0.8000 F1=0.7805 support=80
Macro F1=0.8406  Weighted F1=0.8472  Micro F1=0.8467


## Part 4: Stage 1 error analysis
The weakest class is Neutral. The code below lists where the errors go.

In [3]:
print('Neutral row (actual Neutral):', dict(zip(labels,S1[2].astype(int))), '-> 30 of 80 wrong')
print('Neutral column (predicted Neutral):', dict(zip(labels,S1[:,2].astype(int))), '-> 27 wrongly labelled Neutral')
print('Pos<->Neg confusion:', int(S1[0,1]), 'and', int(S1[1,0]), '(small)')

Neutral row (actual Neutral): {'Positive': np.int64(20), 'Negative': np.int64(10), 'Neutral': np.int64(50)} -> 30 of 80 wrong
Neutral column (predicted Neutral): {'Positive': np.int64(15), 'Negative': np.int64(12), 'Neutral': np.int64(50)} -> 27 wrongly labelled Neutral
Pos<->Neg confusion: 5 and 8 (small)


**Where it fails:** Neutral (F1 about 0.64). Neutral reviews are often pushed to Positive or Negative, and many Positive/Negative reviews are called Neutral. The LLM does not know what 'Neutral' means in this dataset.

**Recommended changes:** define Neutral clearly; add few-shot examples including borderline mixed reviews; add a rule for mixed reviews; ask for a one-sentence justification; force the output to be one of the three label names; use low temperature.

## Part 5: Stage 1 vs Stage 2

In [4]:
rows=[('Accuracy','acc'),('Macro F1','macro'),('Weighted F1','weighted'),('Micro F1','micro')]
print(f"{'Metric':14s}{'Stage1':>8s}{'Stage2':>8s}{'Change':>9s}")
for n,k in rows: print(f'{n:14s}{r1[k]:8.4f}{r2[k]:8.4f}{r2[k]-r1[k]:+9.4f}')
for i,l in enumerate(labels): print(f'{l+" F1":14s}{r1["F"][i]:8.4f}{r2["F"][i]:8.4f}{r2["F"][i]-r1["F"][i]:+9.4f}')
print('Correct predictions: 230 ->', int(np.trace(S2)))
print('CIs overlap:', r2['lo']<r1['hi'])

Metric          Stage1  Stage2   Change
Accuracy        0.7667  0.8467  +0.0800
Macro F1        0.7546  0.8406  +0.0859
Weighted F1     0.7659  0.8472  +0.0812
Micro F1        0.7667  0.8467  +0.0800
Positive F1     0.8065  0.8797  +0.0732
Negative F1     0.8205  0.8615  +0.0410
Neutral F1      0.6369  0.7805  +0.1435
Correct predictions: 230 -> 254
CIs overlap: True


Neutral improved most (recall 0.625 to 0.80). Overall accuracy rose by 8 points. The CIs overlap slightly, so with only 300 reviews the gain is clear but not proven by the intervals alone.

## Part 6: Prompt stability (5 runs)

In [5]:
runs=np.array([0.847,0.840,0.853,0.837,0.850])
print(f'Mean = {runs.mean():.4f}, SD (sample, n-1) = {runs.std(ddof=1):.4f}')
print(f'Result: {runs.mean():.3f} ± {runs.std(ddof=1):.3f}, range {runs.min()} to {runs.max()}')

Mean = 0.8454, SD (sample, n-1) = 0.0067
Result: 0.845 ± 0.007, range 0.837 to 0.853


The SD is under 1 percentage point, so the improved prompt is stable: the order of the examples barely matters.

## Part 7: BLEU-4, ROUGE-1/2/L, METEOR
Both texts are lower-cased with punctuation removed.

In [6]:
ref='the delivery was late but the product quality is good'.split()
cand='the product quality is good but delivery was slow'.split()
def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1))
# BLEU-4 (clipped n-gram precision, brevity penalty, no smoothing needed: all p_n > 0)
ps=[]
for n in range(1,5):
    c,r=ngrams(cand,n),ngrams(ref,n); m=sum((c&r).values()); tot=sum(c.values()); ps.append(m/tot)
    print(f'p{n} = {m}/{tot} = {m/tot:.4f}')
bp=1 if len(cand)>=len(ref) else math.exp(1-len(ref)/len(cand))
bleu=bp*math.exp(sum(math.log(p) for p in ps)/4)
print(f'Brevity penalty = {bp:.4f}; BLEU-4 = {bleu:.4f}')
# ROUGE-N
def rouge_n(n):
    c,r=ngrams(cand,n),ngrams(ref,n); m=sum((c&r).values())
    P=m/sum(c.values()); R=m/sum(r.values()); return P,R,2*P*R/(P+R)
for n in (1,2): print('ROUGE-%d  P=%.4f R=%.4f F1=%.4f'%((n,)+rouge_n(n)))
# ROUGE-L (longest common subsequence)
def lcs(x,y):
    d=[[0]*(len(y)+1) for _ in range(len(x)+1)]
    for i in range(len(x)):
        for j in range(len(y)):
            d[i+1][j+1]=d[i][j]+1 if x[i]==y[j] else max(d[i][j+1],d[i+1][j])
    return d[-1][-1]
L=lcs(cand,ref); P,R=L/len(cand),L/len(ref)
print(f'ROUGE-L  LCS={L} P={P:.4f} R={R:.4f} F1={2*P*R/(P+R):.4f}')
# METEOR (exact matches; alignment chosen to give the fewest chunks)
import itertools
cand_to_ref=[]  # candidate word -> reference positions it can match
for w in cand: cand_to_ref.append([i for i,x in enumerate(ref) if x==w])
best=None
for combo in itertools.product(*[p if p else [None] for p in cand_to_ref]):
    used=[c for c in combo if c is not None]
    if len(set(used))<len(used): continue
    chunks=1; prev=None
    seq=[c for c in combo if c is not None]
    # a chunk continues only if matched words are adjacent in BOTH texts
    chunks=0; prev_i=None; prev_c=None
    for ci,c in enumerate(combo):
        if c is None: prev_i=None; continue
        if prev_i is not None and ci==prev_i+1 and c==prev_c+1: pass
        else: chunks+=1
        prev_i,prev_c=ci,c
    if best is None or chunks<best[0]: best=(chunks,len(used))
chunks,m=best
Pm,Rm=m/len(cand),m/len(ref)
fmean=10*Pm*Rm/(Rm+9*Pm); pen=0.5*(chunks/m)**3
print(f'METEOR: matches={m}, chunks={chunks}, Fmean={fmean:.4f}, penalty={pen:.4f}, METEOR={fmean*(1-pen):.4f}')

p1 = 8/9 = 0.8889
p2 = 5/8 = 0.6250
p3 = 3/7 = 0.4286
p4 = 2/6 = 0.3333
Brevity penalty = 0.8948; BLEU-4 = 0.4750
ROUGE-1  P=0.8889 R=0.8000 F1=0.8421
ROUGE-2  P=0.6250 R=0.5556 F1=0.5882
ROUGE-L  LCS=5 P=0.5556 R=0.5000 F1=0.5263
METEOR: matches=8, chunks=3, Fmean=0.8081, penalty=0.0264, METEOR=0.7868


Word overlap is high (ROUGE-1 0.84, METEOR 0.79). Longer-phrase scores (BLEU-4 0.47, ROUGE-L 0.53) are lower because the LLM reversed the order of the two clauses and said 'slow' instead of 'late'. The meaning is the same, so the metrics understate the quality.

## Part 8: Perplexity
Perplexity = exp( -(1/N) · Σ ln p_i )

In [7]:
p=np.array([0.60,0.40,0.50,0.80,0.70,0.30,0.25,0.50,0.20])
s=np.log(p).sum(); avg=s/len(p)
print(f'Sum ln p = {s:.4f}, average = {avg:.4f}, Perplexity = {math.exp(-avg):.4f}')

Sum ln p = -7.5929, average = -0.8437, Perplexity = 2.3249


Perplexity is about 2.32: on average the model chose between roughly 2 to 3 equally likely words per step, which is low (confident). The least certain tokens were 'slow' (0.20), 'delivery' (0.25) and 'but' (0.30). Perplexity shows model confidence, not correctness, so read it together with the overlap metrics.